In [200]:
import numpy as np

# Termopares
termopares = {
    "K": {"coef": [0.0, 0.039, -0.0001, 0.000002, -0.00000003], "precisao": 1.5, "resolucao": 0.1, "faixa_min": 0, "faixa_max": 1372},
    "J": {"coef": [0.0, 0.05, -0.0002, 0.000003, -0.00000004], "precisao": 2.0, "resolucao": 0.1, "faixa_min": 0, "faixa_max": 760},
    "T": {"coef": [0.0, 0.04, -0.00015, 0.0000025, -0.00000002], "precisao": 1.0, "resolucao": 0.05, "faixa_min": -200, "faixa_max": 350},
    "E": {"coef": [0.0, 0.06, -0.00025, 0.000004, -0.00000005], "precisao": 1.0, "resolucao": 0.05, "faixa_min": -200, "faixa_max": 900}
}

# Seleção do termopar
print("Tipos de termopar disponíveis:")
for letra, dados in termopares.items():
    print(f"{letra}: Faixa {dados['faixa_min']} a {dados['faixa_max']} °C, Precisão ±{dados['precisao']} °C, Resolução {dados['resolucao']} °C")

tipo_termopar = input("Escolha o tipo de termopar (K, J, T, E): ").upper()
if tipo_termopar not in termopares:
    raise ValueError("Termopar inválido!")

# Inserir temperatura nominal do sistema
temperatura_nominal = float(input("Insira a temperatura nominal do sistema (°C), máxima 1250°C: "))
if temperatura_nominal > 1250:
    raise ValueError("Temperatura acima do máximo suportado pelo conversor (1250°C).")
dados_tp = termopares[tipo_termopar]

# Função para calcular a tensão nominal do termopar
def tensao_nominal_termopar(temp, dados_tp):
    coef = dados_tp['coef']
    tensao = sum(c * temp**i for i, c in enumerate(coef))
    return tensao

# Simula erro do termopar
erro_tp = np.random.uniform(-dados_tp['precisao'], dados_tp['precisao'])
tensao_tp_nominal = tensao_nominal_termopar(temperatura_nominal + erro_tp, dados_tp)
print(f"Tensão nominal do termopar ({tipo_termopar}) com erro simulado = {tensao_tp_nominal:.6f} V")

Tipos de termopar disponíveis:
K: Faixa 0 a 1372 °C, Precisão ±1.5 °C, Resolução 0.1 °C
J: Faixa 0 a 760 °C, Precisão ±2.0 °C, Resolução 0.1 °C
T: Faixa -200 a 350 °C, Precisão ±1.0 °C, Resolução 0.05 °C
E: Faixa -200 a 900 °C, Precisão ±1.0 °C, Resolução 0.05 °C
Escolha o tipo de termopar (K, J, T, E): K
Insira a temperatura nominal do sistema (°C), máxima 1250°C: 1000
Tensão nominal do termopar (K) com erro simulado = -28106.625054 V


In [202]:
# Conversores comerciais
conversores = {
    "1": {"nome": "NI 9213 (National Instruments)", "precisao": 0.5, "faixa_min": -200, "faixa_max": 1200, "saida_min": 0.0, "saida_max": 5.0},
    "2": {"nome": "AD8495 (Analog Devices)", "precisao": 1.0, "faixa_min": -40, "faixa_max": 1250, "saida_min": 0.0, "saida_max": 5.0}
}

print("Conversores disponíveis:")
for n, dados in conversores.items():
    print(f"{n}: {dados['nome']}, Faixa {dados['faixa_min']} a {dados['faixa_max']} °C, Saída {dados['saida_min']} a {dados['saida_max']} V, Precisão ±{dados['precisao']} °C")

num_conv = input("Escolha o conversor (1 ou 2): ")
if num_conv not in conversores:
    raise ValueError("Conversor inválido!")

dados_conv = conversores[num_conv]

# Conversor transforma tensão do termopar em saída proporcional
def conversor_comercial(tensao_tp, dados_conv, temp_nominal):
    ganho = (dados_conv['saida_max'] - dados_conv['saida_min']) / (dados_conv['faixa_max'] - dados_conv['faixa_min'])
    offset = dados_conv['saida_min'] - ganho * dados_conv['faixa_min']
    tensao_saida = ganho * temp_nominal + offset
    erro_conv = np.random.uniform(-dados_conv['precisao'], dados_conv['precisao'])
    tensao_saida_com_erro = tensao_saida + ganho * erro_conv
    return tensao_saida_com_erro, ganho, offset

tensao_convertida, ganho_conv, offset_conv = conversor_comercial(tensao_tp_nominal, dados_conv, temperatura_nominal)
print(f"Tensão de saída do conversor ({dados_conv['nome']}) com erro simulado: {tensao_convertida:.3f} V")

Conversores disponíveis:
1: NI 9213 (National Instruments), Faixa -200 a 1200 °C, Saída 0.0 a 5.0 V, Precisão ±0.5 °C
2: AD8495 (Analog Devices), Faixa -40 a 1250 °C, Saída 0.0 a 5.0 V, Precisão ±1.0 °C
Escolha o conversor (1 ou 2): 1
Tensão de saída do conversor (NI 9213 (National Instruments)) com erro simulado: 4.285 V


In [219]:
# Display
nome_display = "TP-500 Display (Omega Engineering)"
precisao_display = 0.5  # ±°C

# Converte tensão do conversor em temperatura
temp_display_ideal = (tensao_convertida - offset_conv) / ganho_conv
erro_display = np.random.uniform(-precisao_display, precisao_display)
temp_display_com_erro = temp_display_ideal + erro_display

print(f"=== Display do Termopar: {nome_display} ===")
print(f"Temperatura mostrada: {temp_display_com_erro:.2f} °C (erro ±{precisao_display} °C)")

=== Display do Termopar: TP-500 Display (Omega Engineering) ===
Temperatura mostrada: 999.51 °C (erro ±0.5 °C)


In [220]:
# Termômetro de referência baseado na mesma temperatura nominal
nome_termometro = "PT100 Classe A (Omega Engineering)"
precisao_termometro = 0.05  # ±°C

def simular_termometro_preciso(temp_nominal, precisao):
    erro = np.random.uniform(-precisao, precisao)
    return temp_nominal + erro

# Leitura do termômetro de referência
temp_referencia = simular_termometro_preciso(temperatura_nominal, precisao_termometro)

# Comparação
print("=== Comparação com Termômetro de Referência ===")
print(f"Valor exibido no display ({nome_display}) = {temp_display_com_erro:.2f} °C")
print(f"Valor do termômetro de referência ({nome_termometro}) = {temp_referencia:.2f} °C")
print(f"Diferença = {abs(temp_display_com_erro - temp_referencia):.2f} °C")

=== Comparação com Termômetro de Referência ===
Valor exibido no display (TP-500 Display (Omega Engineering)) = 999.51 °C
Valor do termômetro de referência (PT100 Classe A (Omega Engineering)) = 1000.04 °C
Diferença = 0.54 °C
